# 01. Dataset Inspection


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import sys

# Ensure project root is in sys.path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import load_manifest

MANIFEST_PATH = ROOT / "data" / "metadata.csv"
print(f"Project root : {ROOT}")
print(f"Manifest path: {MANIFEST_PATH} (exists: {MANIFEST_PATH.exists()})")

Project root : D:\DL
Manifest path: D:\DL\data\metadata.csv (exists: True)


## 1. Load Manifest

In [2]:
df = load_manifest(MANIFEST_PATH, exercise_filter="squat")
print(f"Total squat clips found: {len(df)}")
print(f"Unique participants     : {df['subject_id'].nunique()}")
df.head()

Total squat clips found: 150
Unique participants     : 25


,video_path,label,subject_id,exercise,view
6,data\raw\subject_wise\Dataset Subject-wise\sub...,good,subject_026,squat,diagonal
7,data\raw\subject_wise\Dataset Subject-wise\sub...,good,subject_026,squat,front
8,data\raw\subject_wise\Dataset Subject-wise\sub...,good,subject_026,squat,side
9,data\raw\subject_wise\Dataset Subject-wise\sub...,bad,subject_026,squat,diagonal
10,data\raw\subject_wise\Dataset Subject-wise\sub...,bad,subject_026,squat,front


## 2. Label & View Distributions

In [3]:
view_label_crosstab = df.groupby(["label", "view"]).size().unstack(fill_value=0)
display(view_label_crosstab)

print("\nTotal per label:")
print(df["label"].value_counts())

view,diagonal,front,side
label,,,
bad,25,25,25
good,25,25,25



Total per label:
label
good    75
bad     75
Name: count, dtype: int64


## 3. Video Integrity Checks

In [4]:
missing_files = []
durations, resolutions = [], []

for row in df.itertuples(index=False):
    v_path = ROOT / row.video_path
    if not v_path.exists():
        missing_files.append(str(v_path))
        continue
    
    cap = cv2.VideoCapture(str(v_path))
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    
    durations.append(frames / fps)
    resolutions.append((w, h))

assert len(missing_files) == 0, f"Missing {len(missing_files)} video files!"
print("All 150 video files exist and are readable.")
print(f"Average duration: {np.mean(durations):.2f}s (min: {np.min(durations):.2f}s, max: {np.max(durations):.2f}s)")
print(f"Common resolutions: {set(resolutions)}")

All 150 video files exist and are readable.
Average duration: 29.65s (min: 8.20s, max: 38.67s)
Common resolutions: {(540, 960), (810, 1440), (358, 636), (348, 624)}
